# Brasil nas Olimpíadas 🏅

> Pipeline de dados construído no Microsoft Fabric para analisar a performance do Brasil nos Jogos Olímpicos — 1896 a 2024.

## Sobre o projeto

A partir de um dataset com atletas olímpicos de todo o mundo, filtramos e segmentamos os dados brasileiros para responder perguntas estratégicas sobre medalhas, esportes e atletas ao longo de 128 anos de história olímpica.

## Perguntas que este projeto responde

| # | Dimensão | Pergunta |
|---|----------|----------|
| 01 | Gênero | Qual gênero conquistou mais medalhas pelo Brasil? |
| 02 | Esporte | Qual esporte trouxe mais ouro, prata e bronze? |
| 03 | Ano | Em qual edição o Brasil teve seu melhor desempenho? |
| 04 | Atletas | Quem são nossos top 3 atletas com mais medalhas? |

## Arquitetura Medallion

\```
CSV/Excel  →  [Bronze]  →  [Silver]  →  [Gold]  →  Power BI
               raw          limpeza      agregações
\```

## Tecnologias

- Microsoft Fabric / OneLake
- PySpark + Delta Lake
- Lakehouse (Bronze · Silver · Gold)
- Data Factory (pipelines)

## Status

- [x] Bronze — ingestão do dataset
- [x] Silver — limpeza, tipos, deduplicação (8.500 registros validados)
- [ ] Gold — agregações por Brasil
- [ ] Visualização no Power BI

## Dataset

[olympics_athletes_dataset.csv](./olympics_athletes_dataset.csv) — Atletas Olímpicos 1896–2024


## Análise Exploratória de Dados — Camada Bronze

Antes de qualquer transformação, inspecionamos os dados brutos para entender
a estrutura da tabela: tipos das colunas, volume de registros e qualidade inicial.
Essa etapa é obrigatória para guiar as decisões de limpeza na camada Silver.

## Camada Bronze — Validando os dados brutos

Os dados foram ingeridos via pipeline do Data Factory e armazenados como
tabela Delta no Lakehouse. Aqui carregamos a tabela e exibimos as primeiras
linhas para confirmar que a ingestão foi bem-sucedida.

- Fonte: `olympics_athletes_dataset.csv`
- Tabela: `Bronze_Atletas.Atletas_Olimpicos`
- Ação: leitura e inspeção visual (sem transformações)

In [ ]:
# 1. Instalando o kagglehub
%pip install kagglehub -q

import kagglehub
import pandas as pd
from pyspark.sql import functions as F

# Baixando o dataset
path = kagglehub.dataset_download("stefanydeoliveira/summer-olympics-medals-1896-2024")
print(f"Dataset baixado em: {path}")

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


 53%|█████▎    | 2.00M/3.75M [00:01<00:00, 2.14MB/s]

In [ ]:
# 2. Copiar o CSV para o Lakehouse
import shutil
import os

origem = "/home/trusted-service-user/.cache/kagglehub/datasets/stefanydeoliveira/summer-olympics-medals-1896-2024/versions/1/olympics_dataset.csv"
destino = "/lakehouse/default/Files/olympics_dataset.csv"

shutil.copy(origem, destino)
print("Arquivo copiado para o Lakehouse!")
print(f"Existe no destino: {os.path.exists(destino)}")

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 10, Finished, Available, Finished, False)

Arquivo copiado para o Lakehouse!
Existe no destino: True


In [4]:
# 3. Agora ler do Lakehouse e criar a Bronze:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/olympics_dataset.csv")

spark.sql("CREATE SCHEMA IF NOT EXISTS Bronze_Atletas")
df_bronze.write.format("delta").mode("overwrite") \
    .saveAsTable("Engenharia_Fabric.LH_Olimpiadas.Bronze_Atletas.Atletas_Olimpicos")

print(f"Bronze recriada com sucesso! {df_bronze.count()} registros")
display(df_bronze.limit(5))

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 11, Finished, Available, Finished, False)

Bronze recriada com sucesso! 252565 registros


SynapseWidget(Synapse.DataFrame, c1e921b5-93bd-4cfe-bfba-14df14023ade)

## Camada Silver — Limpeza e padronização

Com os dados brutos validados, aplicamos as seguintes transformações para
garantir qualidade e consistência antes das análises:

1. **Nulos em medalha** → substituídos por `"Nenhuma Medalha"`
2. **Coluna `year`** → convertida para `IntegerType`
3. **Coluna `esporte`** → padronizada em letras maiúsculas com `F.upper()`
4. **Duplicatas** → removidas com `dropDuplicates()`

> Resultado: **8.500 registros únicos** salvos em `Silver_Atletas.Atletas_Olimpicos_Tratado`

In [3]:
#Criando o esquema
spark.sql("CREATE SCHEMA IF NOT EXISTS Silver_Atletas")

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 12, Finished, Available, Finished, False)

DataFrame[]

In [4]:
#Bibliotecas
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

#1. Lendo a camada Bronze
df_bronze = spark.read.table("Engenharia_Fabric.LH_Olimpiadas.Bronze_Atletas.Atletas_Olimpicos")

#2. Limpeza e Padronizar camada Silver
df_silver = df_bronze.select(
    F.col("Name").alias("atleta_nome"),            
    F.when(F.col("Sex") == "M", "Masculino")       
     .when(F.col("Sex") == "F", "Feminino")
     .otherwise("Não informado").alias("genero"),
    F.col("Year").cast(IntegerType()).alias("year"),
    F.col("Team").alias("pais"),                   
    F.col("NOC").alias("noc"),                     # código do país (BRA)
    F.col("Season").alias("temporada"),            # Summer/Winter
    F.col("City").alias("cidade"),                 # cidade sede
    F.upper(F.col("Sport")).alias("esporte"),     
    F.col("Event").alias("evento"),                
    #Tratamento de Medalhas — "No medal"
    F.when(F.col("Medal").isNull(), "Nenhuma Medalha")
     .when(F.col("Medal") == "No medal", "Nenhuma Medalha")
     .when(F.col("Medal") == "Gold", "Ouro")
     .when(F.col("Medal") == "Silver", "Prata")
     .when(F.col("Medal") == "Bronze", "Bronze")
     .otherwise("Nenhuma Medalha").alias("medalha")
).dropDuplicates()

#3. Salvando na Camada Silver com uma nova Tabela Delta
df_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("Engenharia_Fabric.LH_Olimpiadas.Silver_Atletas.Atletas_Olimpicos_Tratado")

print("Tabela Silver criada com sucesso!")
print(f"Total de registros: {df_silver.count()}")
df_silver.select("medalha").distinct().show()

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 13, Finished, Available, Finished, False)

Tabela Silver recriada com sucesso!


Total de registros: 251099
+---------------+
|        medalha|
+---------------+
|          Prata|
|Nenhuma Medalha|
|         Bronze|
|           Ouro|
+---------------+



In [5]:
# 1. Carregar os dados da tabela que criamos
df = spark.read.table("Silver_Atletas.Atletas_Olimpicos_Tratado")

# 2. Ver as primeiras 5 linhas para garantir que está tudo certo
display(df.limit(5))


StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1574631-16ee-4820-953b-043545df70f9)

In [ ]:
# 3. Validando os Tipos de colunas 

print(f"Total de registros: {df.count()}")
print("---")
df.printSchema()

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 15, Finished, Available, Finished, False)

Total de registros: 251099
---
root
 |-- atleta_nome: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- pais: string (nullable = true)
 |-- noc: string (nullable = true)
 |-- temporada: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- esporte: string (nullable = true)
 |-- evento: string (nullable = true)
 |-- medalha: string (nullable = true)



In [ ]:
# 4. Validando se as outras colunas alem da "medalha" possui algum valor nulo

from pyspark.sql import functions as F

nulos = df.select([
    F.sum(F.when(F.col(c).isNull(),1). otherwise(0)).alias(c)
    for c in df.columns
])
display(nulos)

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52f901a2-1fcc-4bcb-a743-c603a4a37efe)

In [ ]:
# 5. Verificar valores distintos

print("=== Valores de Medalha ===")
df.groupBy("medalha").count().orderBy("count", ascending = False).show()

print("=== Valores de Genero ===")
df.groupBy("genero").count().show

print("=== Anos distintos (min/max) ===")
df.select(F.min("year"), F.max("year")).show()

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 17, Finished, Available, Finished, False)

=== Valores de Medalha ===


+---------------+------+
|        medalha| count|
+---------------+------+
|Nenhuma Medalha|212317|
|         Bronze| 13062|
|           Ouro| 12993|
|          Prata| 12727|
+---------------+------+

=== Valores de Genero ===
=== Anos distintos (min/max) ===


+---------+---------+
|min(year)|max(year)|
+---------+---------+
|     1896|     2024|
+---------+---------+



In [ ]:
# 6. Confirmando que realmente esta sem duplicatas

total = df.count()
distintos = df.distinct().count()

print(f"Total de linhas: {total}")
print(f"Linhas distintas: {distintos}")
print(f"Duplicatas restantes: {total - distintos}")

if total == distintos:
    print("Sem duplicatas na Silver!")
else: 
    print("Ainda existem duplicatas")


StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 18, Finished, Available, Finished, False)

Total de linhas: 251099
Linhas distintas: 251099
Duplicatas restantes: 0
Sem duplicatas na Silver!


## Camada Gold — Análises do Brasil

Com os dados da Silver filtrados apenas para o Brasil, respondemos as 4 perguntas
do projeto. Cada análise gera uma tabela Delta pronta para consumo no Power BI.

In [ ]:
# 1. Validando a nomenclatura do Brasil na base

df = spark.read.table("Engenharia_Fabric.LH_Olimpiadas.Silver_Atletas.Atletas_Olimpicos_Tratado")

df.select("pais").distinct().filter(
    F.col("pais").contains("raz") | F.col("pais").contains("BRA")
).show(truncate=False)

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 19, Finished, Available, Finished, False)

+-------------------+
|pais               |
+-------------------+
|Brazil-2           |
|Congo (Brazzaville)|
|Brazil             |
|Brazil-1           |
+-------------------+



In [ ]:
# 2. Segmentando os dados

from pyspark.sql import functions as F 

df = spark.read.table("Engenharia_Fabric.LH_Olimpiadas.Silver_Atletas.Atletas_Olimpicos_Tratado")

# Filtrando apenas o Brasil e apenas quem ganhou a medalha
df_brasil = df.filter(F.col("pais") == "Brazil")
df_medalhas = df_brasil.filter(F.col("medalha") != "Nenhuma Medalha")

spark.sql("CREATE SCHEMA IF NOT EXISTS Gold_Atletas")

print(f"Total de registros do Brasil: {df_brasil.count()}")
print(f"Total de medalhas do Brasil: {df_medalhas.count()}")

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 20, Finished, Available, Finished, False)

Total de registros do Brasil: 4522


Total de medalhas do Brasil: 570


In [ ]:
# 3. Pergunta 01: Gênero com mais medalhas:

gold_genero = df_medalhas.replace({"Female": "Feminino", "Male": "Masculino"}, subset=["genero"]) \
    .groupBy("genero", "medalha") \
    .count() \
    .withColumnRenamed("count", "total") \
    .orderBy("genero", "medalha")
    
display(gold_genero)

gold_genero.write.format("delta").mode("overwrite")\
.saveAsTable("Gold_Atletas.Brasil_Medalhas_Por_Genero")

print("Tabela Gold criada: Brasil_Medalhas_Por_Genero")


StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e28d819f-698c-4a07-a110-f8c3b12cf706)

Tabela Gold criada: Brasil_Medalhas_Por_Genero


In [ ]:
# 4. Pergunta 02: Qual esporte trouxe mais ouro, prata e bronze?

# Dicionário de tradução dos esportes
traducao_esportes = {
    "BOXING": "Boxe",
    "BOBSLED": "Bobsled",
    "FIGURE SKATING": "Patinação Artística",
    "SKI JUMPING": "Salto de Esqui",
    "FREESTYLE SKIING": "Esqui Estilo Livre",
    "CANOE/KAYAK": "Canoagem",
    "GYMNASTICS (ARTISTIC)": "Ginástica Artística",
    "CROSS-COUNTRY SKIING": "Esqui Cross-Country",
    "SPEED SKATING": "Patinação de Velocidade",
    "CYCLING": "Ciclismo",
    "SWIMMING": "Natação",
    "WEIGHTLIFTING": "Levantamento de Peso",
    "BASKETBALL": "Basquete",
    "ATHLETICS": "Atletismo",
    "ALPINE SKIING": "Esqui Alpino",
    "LUGE": "Luge",
    "BIATHLON": "Biatlo",
    "ROWING": "Remo",
    "VOLLEYBALL": "Vôlei",
    "FOOTBALL": "Futebol",
    "JUDO": "Judô",
    "SAILING": "Vela",
    "SHOOTING": "Tiro",
    "TRIATHLON": "Triatlo",
    "TENNIS": "Tênis",
    "TAEKWONDO": "Taekwondo",
    "WRESTLING": "Luta Livre",
    "HANDBALL": "Handebol",
    "HOCKEY": "Hóquei",
    "ARCHERY": "Tiro com Arco",
    "EQUESTRIAN": "Hipismo",
    "FENCING": "Esgrima",
    "DIVING": "Saltos Ornamentais",
    "WATER POLO": "Polo Aquático",
    "BEACH VOLLEYBALL": "Vôlei de Praia",
    "SKELETON": "Skeleton",
    "CURLING": "Curling",
    "ICE HOCKEY": "Hóquei no Gelo",
    "SHORT TRACK SPEED SKATING": "Patinação de Velocidade em Pista Curta",
    "SYNCHRONIZED SWIMMING": "Nado Sincronizado",
    "MODERN PENTATHLON": "Pentatlo Moderno",
    "BADMINTON": "Badminton",
    "TABLE TENNIS": "Tênis de Mesa",
    "SOFTBALL": "Softball",
    "BASEBALL": "Beisebol",
    "RUGBY": "Rúgbi",
    "GOLF": "Golfe",
    "SURFING": "Surfe",
    "SKATEBOARDING": "Skate",
    "SPORT CLIMBING": "Escalada Esportiva",
}

# Validando o esporte por agrupamento
gold_esporte = df_medalhas.replace(traducao_esportes, subset=["esporte"]) \
    .groupBy("esporte", "medalha") \
    .count() \
    .withColumnRenamed("count", "total") \
    .orderBy(F.col("total").desc())

display(gold_esporte)

gold_esporte.write.format("delta").mode("overwrite") \
    .saveAsTable("Gold_Atletas.Brasil_Medalhas_Por_Esporte")

print("Tabela Gold criada: Brasil_Medalhas_Por_Esporte")


StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21ca4140-d68c-4e36-a9a6-5f2c1adf97a7)

Tabela Gold criada: Brasil_Medalhas_Por_Esporte


In [ ]:
# 5. Pergunta 03: Em qual edição o Brasil teve seu melhor desempenho?

gold_ano = df_medalhas.groupBy("year") \
    .count() \
    .withColumnRenamed("count", "total_medalhas") \
    .orderBy(F.col("total_medalhas").desc())

display(gold_ano)

gold_ano.write.format("delta").mode("overwrite") \
    .saveAsTable("Gold_Atletas.Brasil_Medalhas_Por_Ano")

print("Tabela Gold criada: Brasil_Medalhas_Por_Ano")

StatementMeta(, 6547465c-b6a6-4e5f-8eea-9474b86ab05e, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 58ff0a20-3cc7-440c-a93b-80ceb527626b)

Tabela Gold criada: Brasil_Medalhas_Por_Ano
